# Experiment 2 — Structured Decomposition (Qwen)

**Model:** `qwen/qwen3.8-27b` via Groq API  
**Benchmark:** Same fixed 250-question obfuscated benchmark used in Experiment 1.

## Fixed benchmark
- FOL / obfuscation: 50
- Blood Relation / obfuscation_l1: 50
- Blood Relation / obfuscation_l2: 50
- Direction Sense / obfuscation: 50
- Number Series / obfuscation: 50
- **Total: 250**

## Method
Structured decomposition: explicitly identify the relevant entities/relations or steps, organize them sequentially, solve from that structured representation, and return only the final answer.

The same 250 questions are used as Experiment 1; only the prompting method changes.

## Safety protocol
First run `TEST_MODE = True`, which evaluates 5 questions (one per variant). After verifying the pipeline, change it to `False` for the full 250-question run.

In [ ]:


!pip -q install pandas openai tqdm


In [ ]:


import os
import re
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from IPython.display import display

# True  = 5-question safety test
# False = full 250-question inference
TEST_MODE = True
TEST_SAMPLES_PER_VARIANT = 1

MODEL_NAME = "qwen/qwen3.8-27b"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

TEMPERATURE = 0
MAX_TOKENS = 500

CHECKPOINT_EVERY = 10

print("Model:", MODEL_NAME)
print("TEST_MODE:", TEST_MODE)


In [ ]:


GROQ_API_KEY = None

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass("Enter your GROQ_API_KEY: ")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is required.")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL
)

print("Groq client ready.")


In [ ]:


from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

csv_names = [
    name for name in uploaded.keys()
    if name.lower().endswith(".csv")
]

if not csv_names:
    raise ValueError("Please upload the 250-question CSV.")

csv_name = csv_names[0]
df = pd.read_csv(csv_name)

print("Loaded:", csv_name)
print("Rows:", len(df))
print("Columns:", list(df.columns))

display(df.head())


In [ ]:


required_columns = [
    "dataset",
    "variant",
    "prompt",
    "ground_truth"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

expected = {
    ("FOL", "obfuscation"): 50,
    ("Blood_Relation", "obfuscation_l1"): 50,
    ("Blood_Relation", "obfuscation_l2"): 50,
    ("Direction_Sense", "obfuscation"): 50,
    ("Number_Series", "obfuscation"): 50,
}

distribution = (
    df.groupby(["dataset", "variant"])
      .size()
      .reset_index(name="count")
)

display(distribution)

for (dataset, variant), expected_count in expected.items():
    actual_count = int(
        (
            (df["dataset"].astype(str) == dataset) &
            (df["variant"].astype(str) == variant)
        ).sum()
    )

    print(
        f"{dataset} / {variant}: "
        f"{actual_count} / {expected_count}"
    )

    if actual_count != expected_count:
        raise ValueError(
            f"Unexpected count for {dataset}/{variant}: "
            f"{actual_count}; expected {expected_count}."
        )

if len(df) != 250:
    raise ValueError(
        f"Expected 250 total rows, found {len(df)}."
    )

print("\n✅ Fixed 250-question benchmark validated.")


In [ ]:

STRUCTURED_DECOMPOSITION_SYSTEM = r'''
You are solving an obfuscated logical reasoning problem.

Use STRUCTURED DECOMPOSITION before solving.

1. IDENTIFY
   Identify every relevant entity, fact, relation, condition, quantity,
   movement, ordering, or sequence element in the problem.

2. STRUCTURE
   Convert the information into an explicit structured representation
   appropriate to the task.

   For relationship problems:
   - identify the people/entities
   - extract each relationship
   - connect the relationships into a chain or structure
   - preserve relationship direction

   For direction problems:
   - identify the starting point
   - list each movement in order
   - represent movements as directions/relative changes
   - combine them to determine the final position

   For number/sequence problems:
   - list the relevant sequence values
   - identify candidate transformations or patterns
   - verify the pattern across the available values

   For logical/FOL problems:
   - list the relevant entities, predicates, facts, and constraints
   - connect them according to the stated conditions

3. SOLVE
   Solve the question from the structured representation.
   Do not add facts that are not present.

4. VERIFY
   Check that the final answer is consistent with the original problem.

FINAL:
Return ONLY the final answer requested by the question.
Do not output the decomposition, reasoning, explanation, labels, or extra text.
'''

print("Structured Decomposition prompt ready.")


In [ ]:


def normalize_answer(value):
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip()

    prefixes = [
        "Final Answer:",
        "Final answer:",
        "Answer:",
        "answer:",
        "The answer is:",
        "the answer is:",
    ]

    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
            break

    text = re.sub(r"\s+", " ", text).strip().lower()
    text = text.strip(" 	\n\r\"'.,;:!?")

    return text


def run_model(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": STRUCTURED_DECOMPOSITION_SYSTEM
            },
            {
                "role": "user",
                "content": str(prompt)
            }
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    content = response.choices[0].message.content

    if content is None:
        return ""

    return str(content).strip()


print("Model and evaluation helpers ready.")


In [ ]:


if TEST_MODE:

    parts = []

    variant_order = [
        ("FOL", "obfuscation"),
        ("Blood_Relation", "obfuscation_l1"),
        ("Blood_Relation", "obfuscation_l2"),
        ("Direction_Sense", "obfuscation"),
        ("Number_Series", "obfuscation"),
    ]

    for dataset_name, variant_name in variant_order:

        subset = df[
            (df["dataset"].astype(str) == dataset_name) &
            (df["variant"].astype(str) == variant_name)
        ]

        if len(subset) < TEST_SAMPLES_PER_VARIANT:
            raise ValueError(
                f"Not enough rows for {dataset_name}/{variant_name}."
            )

        parts.append(
            subset.head(TEST_SAMPLES_PER_VARIANT)
        )

    run_df = pd.concat(parts, ignore_index=True)

    print("TEST MODE")
    print("Rows to process:", len(run_df))

else:

    run_df = df.copy().reset_index(drop=True)

    print("FULL MODE")
    print("Rows to process:", len(run_df))

    if len(run_df) != 250:
        raise ValueError(
            f"Expected 250 rows, found {len(run_df)}."
        )

display(
    run_df[
        ["dataset", "variant", "row_number", "response_id", "ground_truth"]
    ]
)


In [ ]:


results = []

if TEST_MODE:

    checkpoint_file = None

else:

    checkpoint_file = (
        "experiment_02_structured_decomposition_Qwen_250_checkpoint.csv"
    )

    if os.path.exists(checkpoint_file):

        previous = pd.read_csv(checkpoint_file)
        results = previous.to_dict("records")

        processed_ids = set(
            previous["response_id"]
            .astype(str)
            .tolist()
        )

        print(
            f"Found checkpoint with {len(previous)} completed rows."
        )

    else:

        processed_ids = set()

        print(
            "No previous full-run checkpoint found. "
            "Starting from row 1."
        )

print("\nStarting inference...")

for i, (_, row) in enumerate(
    tqdm(
        run_df.iterrows(),
        total=len(run_df),
        desc="Qwen Structured Decomposition"
    ),
    start=1
):

    response_id = str(row.get("response_id", ""))

    if not TEST_MODE and response_id in processed_ids:
        continue

    prediction = ""
    error = ""

    max_attempts = 3

    for attempt in range(1, max_attempts + 1):

        try:

            prediction = run_model(row["prompt"])
            break

        except Exception as e:

            error = repr(e)

            if attempt < max_attempts:

                wait_seconds = 2 ** (attempt - 1)

                print(
                    f"\nAPI error on {response_id} "
                    f"(attempt {attempt}/{max_attempts}): {e}"
                )

                print(
                    f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)

    ground_truth = normalize_answer(
        row["ground_truth"]
    )

    prediction_clean = normalize_answer(
        prediction
    )

    correct = (
        bool(ground_truth)
        and bool(prediction_clean)
        and prediction_clean == ground_truth
    )

    results.append({
        "dataset": row["dataset"],
        "variant": row["variant"],
        "setting": row.get("setting", ""),
        "row_number": row.get("row_number", ""),
        "response_id": response_id,
        "ground_truth": ground_truth,
        "prediction": prediction_clean,
        "correct": correct,
        "error": error,
    })

    if not TEST_MODE and len(results) % CHECKPOINT_EVERY == 0:

        checkpoint_df = pd.DataFrame(results)

        checkpoint_df.to_csv(
            checkpoint_file,
            index=False
        )

        print(
            f"\nCheckpoint saved: "
            f"{len(results)}/250"
        )

    time.sleep(0.2)

results_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("INFERENCE COMPLETE")
print("=" * 60)

print(
    "Rows in results:",
    len(results_df)
)

print(
    "Successful predictions:",
    results_df["prediction"]
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print(
    "Errors:",
    results_df["error"]
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)


In [ ]:


display(
    results_df[
        [
            "dataset",
            "variant",
            "setting",
            "row_number",
            "ground_truth",
            "prediction",
            "correct",
            "error"
        ]
    ]
)


In [ ]:


valid = results_df[
    results_df["ground_truth"].astype(str).str.strip().ne("") &
    results_df["prediction"].astype(str).str.strip().ne("")
].copy()

if len(valid) == 0:
    raise ValueError(
        "No valid prediction/ground-truth pairs available."
    )

overall_accuracy = valid["correct"].mean() * 100

print(
    f"Overall accuracy: {overall_accuracy:.2f}%"
)

variant_summary = (
    valid
    .groupby(["dataset", "variant"])["correct"]
    .agg(
        total="count",
        correct="sum",
        accuracy="mean"
    )
    .reset_index()
)

variant_summary["accuracy_percent"] = (
    variant_summary["accuracy"] * 100
)

variant_summary = variant_summary[
    [
        "dataset",
        "variant",
        "total",
        "correct",
        "accuracy_percent"
    ]
]

print("\nTask + Variant results:")
display(variant_summary)


In [ ]:


if "setting" in valid.columns:

    setting_summary = (
        valid
        .groupby(
            ["dataset", "variant", "setting"]
        )["correct"]
        .agg(
            total="count",
            correct="sum",
            accuracy="mean"
        )
        .reset_index()
    )

    setting_summary["accuracy_percent"] = (
        setting_summary["accuracy"] * 100
    )

    setting_summary = setting_summary[
        [
            "dataset",
            "variant",
            "setting",
            "total",
            "correct",
            "accuracy_percent"
        ]
    ]

    display(setting_summary)


In [ ]:
# 13. Save final results

if TEST_MODE:

    results_file = (
        "experiment_02_structured_decomposition_Qwen_TEST_results.csv"
    )

    summary_file = (
        "experiment_02_structured_decomposition_Qwen_TEST_summary.csv"
    )

else:

    results_file = (
        "experiment_02_structured_decomposition_Qwen_250_results.csv"
    )

    summary_file = (
        "experiment_02_structured_decomposition_Qwen_250_summary.csv"
    )

results_df.to_csv(
    results_file,
    index=False
)

variant_summary.to_csv(
    summary_file,
    index=False
)

print("Saved:", results_file)
print("Saved:", summary_file)


In [ ]:


from google.colab import files

files.download(results_file)

time.sleep(2)

files.download(summary_file)

print("✅ Download started.")
